Downloading Libraries and Imports

In [1]:
!pip install -q --upgrade "pillow<11.0.0" transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)


Setting up environment

In [2]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_3,
   preprocess_function,
   get_patch_reordering_indices,
   reorder_chessboard_image,
   finetune_and_push_chessboard_model,
)

print("All custom modules and eval utilities imported successfully!")

Cloning repository from https://github.com/Aivon99/BigDataAndTextMiningProject.git...
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: Tesla T4
All custom modules and eval utilities imported successfully!


Dowloading dataset for task 1 from HuggingFace repo

In [3]:
print("Verifying Authentication to Hugging Face...")
notebook_login()
from huggingface_hub import snapshot_download

dataset_name = "bdatm-project/dataset_task3"

# 1. Download the entire dataset repository locally into 'my_dataset_files'
print("Downloading dataset snapshot locally...")
local_data_dir = snapshot_download(
    repo_id=dataset_name,
    repo_type="dataset",
    local_dir="./my_dataset_files"
)

# 2. Load the dataset structure via load_dataset
print(f"Loading dataset '{dataset_name}'...")
dataset_task3 = load_dataset(dataset_name)

# 3. Loop through each split (train, validation, test) and map the second frame correctly
print("Mapping the dataset across splits to include image_t1...")
for split_name in dataset_task3.keys():

    def add_second_frame(example):
        # Build the path incorporating the split folder (train/val/test)
        file_path = os.path.join("./my_dataset_files", split_name, example["file_name_t1"])
        example["image_t1"] = Image.open(file_path).convert("RGB")
        return example

    dataset_task3[split_name] = dataset_task3[split_name].map(add_second_frame)

print("\nDone! Every split has been successfully mapped with image_t1.")
print(dataset_task3)

Verifying Authentication to Hugging Face...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 84 files:   0%|          | 0/84 [00:00<?, ?it/s]

Loading dataset 'bdatm-project/dataset_task3'...


Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

...

Mapping the dataset across splits to include image_t1...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]


Done! Every split has been successfully mapped with image_t1.
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1', 'image_t1'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1', 'image_t1'],
        num_rows: 4
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1', 'image_t1'],
        num_rows: 4
    })
})


In [4]:
# ==========================================
# Verification Script for Task 3 Dataset
# ==========================================

print("Verifying dataset_task3 structure...")

# 1. Check if it's a DatasetDict and list available splits
print(f"Dataset splits available: {list(dataset_task3.keys())}")

# 2. Pick the train split (or any other split) to inspect
sample_split = dataset_task3["train"]
print(f"Number of samples in train split: {len(sample_split)}")

# 3. Grab the first sample
sample = sample_split[0]

# 4. Print core metadata fields
print("\n--- Sample Metadata ---")
print(f"Sample ID: {sample.get('sample_id', 'N/A')}")
print(f"Task type: {sample.get('task', 'N/A')}")
print(f"Ground Truth Move (Target): {sample.get('target', 'N/A')}")
print(f"Prompt preview:\n{sample.get('prompt', 'N/A')[:150]}...")

# 5. Verify images (State t and State t+1)
print("\n--- Image Verification ---")
img_t = sample.get("image")
img_t1 = sample.get("image_t1")

print(f"Frame 1 ('image'): {type(img_t)}")
if img_t is not None:
    print(f"  -> Size: {img_t.size}, Mode: {img_t.mode}")

print(f"Frame 2 ('image_t1'): {type(img_t1)}")
if img_t1 is not None:
    print(f"  -> Size: {img_t1.size}, Mode: {img_t1.mode}")

# 6. Assertion check to ensure everything is ready for training
assert img_t is not None, "Error: Frame 1 ('image') is missing or None!"
assert img_t1 is not None, "Error: Frame 2 ('image_t1') is missing or None!"
assert "target" in sample, "Error: Target field is missing!"

print("\nVerification Successful! dataset_task3 is correctly formatted and ready for training/evaluation.")

Verifying dataset_task3 structure...
Dataset splits available: ['train', 'validation', 'test']
Number of samples in train split: 32

--- Sample Metadata ---
Sample ID: sample_000000
Task type: task3
Ground Truth Move (Target): Qe8
Prompt preview:
You are a specialized model for chessboard temporal reasoning.
Your goal is to identify the move that transitioned the chessboard from State t to Stat...

--- Image Verification ---
Frame 1 ('image'): <class 'PIL.PngImagePlugin.PngImageFile'>
  -> Size: (512, 512), Mode: RGB
Frame 2 ('image_t1'): <class 'PIL.PngImagePlugin.PngImageFile'>
  -> Size: (512, 512), Mode: RGB

Verification Successful! dataset_task3 is correctly formatted and ready for training/evaluation.


## Vanilla Model

Loading model

In [5]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

Loading model Qwen/Qwen3.5-0.8B...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

...

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model and Processor loaded correctly!


Test with baseline on a single sample

In [5]:
from PIL import Image
from huggingface_hub import hf_hub_download

# Grab the first test sample from the Task 3 dataset
test_sample = dataset_task3["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_move = test_sample["target"]  # Task 3 target is the SAN move
sample_id = test_sample["sample_id"]

# Frame 1: State t
frame_t_image = test_sample["image"]

# Frame 2: State t+1

frame_t_plus_1_image = test_sample["image_t1"]

print(f"Sample ID: {sample_id}")
print(f"FEN State: {fen}")
print(f"\n Prompt provided to the model:\n{task_prompt}\n")
print(f"Real Move (Ground Truth): {ground_truth_move}\n")

# Prepare the multimodal input format for the model with two ordered images (State t and State t+1)
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": frame_t_image},
            {"type": "image", "image": frame_t_plus_1_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and pass both images as a list to the processor, then move to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=[frame_t_image, frame_t_plus_1_image],
    padding=True,
    return_tensors="pt"
).to(device)

# Generate the zero-shot prediction for the delta move
print("Waiting for the model to respond....")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_move_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"\n MODEL OUTPUT: \n{predicted_move_string.strip()}")

Sample ID: sample_000000
FEN State: 4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - - 5 28

 Prompt provided to the model:
You are a specialized model for chessboard temporal reasoning.
Your goal is to identify the move that transitioned the chessboard from State t to State t+1.
Input:
- Frame 1: Chessboard state at time t.
- Frame 2: Chessboard state at time t+1.
Output Format:
Return only the move in Standard Algebraic Notation (SAN).

Real Move (Ground Truth): Kg5

Waiting for the model to respond....

 MODEL OUTPUT: 
The move is c4.


Testing vanilla model on the whole dataset using the function *evaluate_chessboard_model_task_1*

In [6]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_3(
    model=model_vanilla,
    processor=processor,
    dataset_split=dataset_task3["test"],
    model_name="Vanilla Qwen3.5-0.8B (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head())

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)

Evaluating Vanilla Qwen3.5-0.8B (Zero-Shot) on Task 3: 100%|██████████| 4/4 [01:03<00:00, 15.84s/it]


Evaluation completed for Vanilla Qwen3.5-0.8B (Zero-Shot) on Task 3! Results saved to task3_vanilla_qwen3.5-0.8b_(zero-shot)_results.csv.

Example of results obtained form the evaluation:


,sample_id,ground_truth,predicted,exact_match
0,sample_000000,Kg5,The move is c4.,0
1,sample_000001,Nxd4,The move is c1.,0
2,sample_000002,Rxb2,The move is c4.,0
3,sample_000003,Nxe6,The move is c2.,0



Comparative Summary DataFrame (all_models_results):


,model_name,exact_match
0,Vanilla Qwen3.5-0.8B (Zero-Shot),0.0


## Vanilla + LoRA

Preprocessing dataset and finetuning

In [13]:
# 1. Preprocessing dataset for Task 3 (Dual-Image Delta Move)
print("Applying preprocessing to Task 3 datasets...")
tokenized_train = dataset_task3["train"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task3["train"].column_names,
)
tokenized_val = dataset_task3["validation"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task3["validation"].column_names,
)

# 2. Configure PEFT and LoRA parameters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to model vanilla and saving the new one into lora_model
lora_model = get_peft_model(model_vanilla, peft_config)
lora_model.print_trainable_parameters()


data_collator = Qwen35DataCollator(processor=processor)

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen_task3_lora_output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=2,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    remove_unused_columns=False, # Essenziale per i modelli multimodali
    report_to="none",
)

# 4. Initialize the Trainer using 'lora_model'
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# 5. Start Fine-Tuning
print("Starting LoRA Supervised Fine-Tuning for Task 3...")
trainer.train()


Applying preprocessing to Task 3 datasets...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435
Starting LoRA Supervised Fine-Tuning for Task 3...


ValueError: Image features and image tokens do not match, tokens: 512, features: 256

Saving model on hugging face

In [ ]:
# 6. Save weights to Hugging Face folder for Task 3
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task3-standard-lora"

print(f"Pushing standard LoRA model and processor to Hugging Face Hub for Task 3: {repo_id_standard}...")

trainer.model.push_to_hub(
    repo_id_standard,
    commit_message="Training complete for standard LoRA baseline on Task 3 (Dual-Image Delta Move)"
)
processor.push_to_hub(
    repo_id_standard
)

print("Fine-tuning completed and weights successfully uploaded to Hugging Face Hub for Task 3!")

Loading model and evaluation

In [ ]:
# 7. Loading Model from Hugging Face and evaluate model using predefined functions for Task 3
print("\nLoading standard LoRA model from Hugging Face for Task 3 evaluation...")

# Load fresh base model instance
base_eval_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Load the LoRA weights directly from the Hub repository
standard_lora_eval_model = PeftModel.from_pretrained(base_eval_model, repo_id_standard)

print("Evaluating the Hugging Face LoRA model on the Task 3 test set...")
lora_results_df, lora_summary_df = evaluate_chessboard_model_task_3(
    model=standard_lora_eval_model,
    processor=processor,
    dataset_split=dataset_task3["test"],
    model_name="Qwen + LoRA Fine-Tuning (from HF)"
)

# Append the new metrics to the global comparison DataFrame
all_models_results = pd.concat([all_models_results, lora_summary_df], ignore_index=True)

print("\nUpdated Comparative Summary Table (all_models_results):")
display(all_models_results)

## Reordering patches - Advanced models

Checking function *get_patch_reordering_indices()*

In [ ]:
for strat in ["raster", "zigzag", "spiral", "file_wise"]:
    order_map = get_patch_reordering_indices(strategy=strat)
    print(f"Strategy '{strat}' first 10 patch indices: {order_map[:10]}")

Let's evalaute the model we created before using three different patch ordering: zigzag, spiral and file-wise. We'll use the function ***reorder_chessboard_image*** defined in *src/eval/utilities.py*

In [ ]:
# ==========================================
# Training-Free Benchmark Loop for REOrder (Task 3)
# ==========================================

# Define the strategies you want to benchmark (as outlined in the project specs)
strategies_to_test = ["zigzag", "spiral", "file_wise"]

# Choose the model to test (we use the fine-tuned LoRA model)
model_to_evaluate = lora_model  # You can switch to 'model_vanilla' if you want to test the vanilla version

for strat in strategies_to_test:
    print(f"\nEvaluating strategy (Training-Free) on Task 3: {strat.upper()}...")

    # 1. Apply the reordering function to BOTH images in the test set (State t and State t+1)
    reordered_test_split = dataset_task3["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8),
            "image_t1": reorder_chessboard_image(sample["image_t1"], strategy=strat, grid_size=8)
        }
    )

    # 2. Run the Task 3 evaluation function on the reordered test split
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_3(
        model=model_to_evaluate,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - TF)"
    )

    # 3. Append the results to your global comparison table
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table for Task 3 (Including Training-Free Strategies) ---")
display(all_models_results)

As highlighted in the summary table, applying unconventional patch reordering strategies (such as Zigzag, Spiral, or File-wise) in a "Training-Free" (TF) manner leads to a performance drop, resulting in an increased Character Error Rate (CER) and Levenshtein distance compared to the standard raster-scan baseline.

To truly reap the benefits of the REOrder methodology, we must proceed with Supervised Fine-Tuning (SFT) directly on the pre-reordered dataset. This will allow the model to adapt its weights and attention layers to the new spatial serialization strategy.

Finetuning the new models on the dataset using function **finetune_and_push_chessboard_model()** defined in *src/eval/utilities.py*. This function directly upload the models on hugging face

In [ ]:
strategies_to_train = ["zigzag", "spiral", "file_wise"]

# Dictionary to store the trained models in memory
trained_reordered_models = {}

for strat in strategies_to_train:
    trained_reordered_models[strat] = finetune_and_push_chessboard_model(
        strategy_name=strat,
        dataset=dataset_task3,
        processor=processor,
        peft_config=peft_config,
        task="task3",
    )

print("\nAll reordered models have been successfully trained and pushed to Hugging Face!")

Evaluating models using function **evaluate_chessboard_model_task_1()** defined in *src/eval/utilities.py*.

Models are downloaded from the hugging face repo.

In [ ]:
# ==========================================
# Evaluator Loop for REOrder SFT Models (Task 3)
# ==========================================

strategies_to_evaluate = ["zigzag", "spiral", "file_wise"]
hf_org_prefix = "bdatm-project"  # Ensure this matches the prefix used for the push

for strat in strategies_to_evaluate:
    print(f"\nLoading and evaluating SFT model for strategy on Task 3: {strat.upper()} from Hugging Face...")

    # 1. Load fresh base model instance
    base_eval_model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3.5-0.8B",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # 2. Load the specific LoRA weights from Hugging Face Hub for Task 3
    repo_id_source = f"{hf_org_prefix}/qwen-task3-{strat}-lora"
    model_to_eval = PeftModel.from_pretrained(base_eval_model, repo_id_source)

    # 3. Apply the specific patch reordering to BOTH images in the test split (State t and State t+1)
    reordered_test_split = dataset_task3["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8),
            "image_t1": reorder_chessboard_image(sample["image_t1"], strategy=strat, grid_size=8)
        }
    )

    # 4. Run the Task 3 evaluation utility function
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_3(
        model=model_to_eval,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - SFT)"
    )

    # 5. Append results to the global comparison DataFrame
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table for Task 3 (Loaded from Hub & Evaluated) ---")
display(all_models_results)

Displaying oredered results

In [ ]:
sorted_results_df = all_models_results.sort_values(by="exact_match", ascending=False).reset_index(drop=True)

print("\n--- Comparative Summary Table for Task 3 (Ordered from Best to Worst by Exact Match) ---")
display(sorted_results_df)